# NexusRAG Quickstart

Ingest a sample document, ask questions, inspect cited passages, and manage the local collection.

Before opening this notebook, install from the repository root with `python -m pip install -e ".[eval]"`. Use that environment as the notebook kernel. Start Ollama and pull the configured model (`ollama pull llama3.2:3b` by default).

The document ships with the repository; embedding models can download on first use, and querying requires local Ollama. This is not a download-free workflow. Runtime settings come from environment variables and `.env`; see `.env.example`.


In [ ]:
import os
from pathlib import Path

# Resolve paths before loading configuration; works from root or notebooks/.
cwd = Path.cwd().resolve()
repo_root = next(
    (
        path
        for path in (cwd, *cwd.parents)
        if (path / "pyproject.toml").is_file() and (path / "examples/sample_paper.md").is_file()
    ),
    None,
)
if repo_root is None:
    raise RuntimeError("Start this notebook inside the NexusRAG repository.")
os.chdir(repo_root)

from scinexusrag.config import get_settings  # noqa: E402
from scinexusrag.pipeline import NexusRAG  # noqa: E402

print(f"Repository: {repo_root}")

## 1. Initialize

Components load when accessed. The default configuration stores documents and indexes under the repository's `data/` directory; override `DATA_DIR` and `LANCEDB_PATH` for a separate collection.


In [ ]:
settings = get_settings()
rag = NexusRAG(settings)

print(f"LLM model: {settings.llm.model}")
print(f"Embedding model: {settings.embedding.model}")
print(f"Storage path: {settings.storage.lancedb_path}")

## 2. Ingest a document

PDF, DOCX, UTF-8 text and Markdown are supported. Ingestion parses text, splits chunks, embeds them and stores them for retrieval. The tracked sample avoids a document download; the embedding model may still need downloading.


In [ ]:
result = rag.ingest(repo_root / "examples/sample_paper.md")

print(f"Document ID: {result.document_id}")
print(f"Chunks created: {result.chunk_count}")
print(f"Word count: {result.word_count}")
print(f"Success: {result.success}")
if not result.success:
    raise RuntimeError(result.error or "Sample ingestion failed.")

## 3. Query

Hybrid retrieval combines dense and sparse scores. When enabled, confidence-gated correction retries retrieval before local Ollama generates an answer. Check the cited passages: retrieval confidence does not establish answer accuracy.


In [ ]:
response = rag.query("What are the main findings of the study?")

print(f"Answer: {response.answer}\n")
print(f"Confidence: {response.confidence:.2f}")
print(f"Processing time: {response.processing_time_ms:.0f}ms")
print(f"\nSources ({len(response.sources)}):")
for i, source in enumerate(response.sources, 1):
    print(f"  [{i}] {source.document_name} (p.{source.page_number}, score: {source.score:.3f})")
    print(f"      {source.content[:100]}...")

## 4. Manage Documents

In [ ]:
docs = rag.list_documents()
for doc in docs:
    name = doc.get("original_filename") or doc.get("filename", "Unknown")
    print(f"  {doc['id'][:8]}... | {name} | {doc.get('word_count', 0)} words")

stats = rag.get_stats()
print(f"\nTotal documents: {stats.total_documents}")
print(f"Total chunks: {stats.total_chunks}")
print(f"LLM available: {stats.llm_available}")

In [ ]:
# Uncomment only when ready to remove the sample from this collection.
# rag.delete_document(result.document_id)

# Removes every document and chunk in the configured collection:
# rag.clear_all()


## 5. Inspect and reproduce benchmarks

Published scores are under `outputs/results/`. The next cell reads them without running models.

From the repository root:

```bash
python -m scinexusrag.eval --sample
python -m scinexusrag.eval
python -m scinexusrag.eval.report --results outputs/generated
```

Sample mode uses packaged datasets, but uncached models may download; its scores do not replace the full benchmark. Full mode requires the pinned datasets and models and fails if they are unavailable. Fresh scores go to `outputs/generated/`; review before publishing. The report reads `outputs/results/` unless `--results` is supplied and writes its bundle under `outputs/generated/paper/`.

These evaluations measure retrieval and evidence detection, not generated-answer accuracy. Generation and RAGTruth evaluations are separate. See `CONTRIBUTING.md` for checks and configuration.


In [ ]:
import json

results_path = repo_root / "outputs/results/scifact_test.json"
res = json.loads(results_path.read_text())
print(f"SciFact ablation ({res['num_queries']} claims, {res['corpus_size']} docs)")
for name, scores in res["systems"].items():
    print(f"  {name:20s} nDCG@10 = {scores['means']['nDCG@10']:.3f}")